## Salinas BESS Monthly Invoice Calculator

This notebook follows the same workflow as the prior PPOA calculator: load inputs, validate them, run the contract calculations, write outputs, and inspect the results.

In [1]:
# Imports
import sys
print(sys.executable)
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Import/reload local modules in dependency order.
module_names = [
    "classes",
    "calculations",
    "data_reader",
    "data_writer",
    "error_checks",
    "report",
    "compensation_calculator",
]

for module_name in module_names:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

calculations = sys.modules["calculations"]
compensation_calculator = sys.modules["compensation_calculator"]
data_reader = sys.modules["data_reader"]
data_writer = sys.modules["data_writer"]
error_checks = sys.modules["error_checks"]
report = sys.modules["report"]

FORCE_MAJEURE_WAITING_PERIOD_HOURS = calculations.DEFAULT_FORCE_MAJEURE_WAITING_PERIOD_HOURS
GRID_SYSTEM_WAITING_PERIOD_HOURS = calculations.DEFAULT_GRID_SYSTEM_WAITING_PERIOD_HOURS
SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS = calculations.DEFAULT_SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
calculate_monthly_results = compensation_calculator.calculate_monthly_results
load_bess_inputs = data_reader.load_bess_inputs
load_monthly_performance_guarantee_inputs = data_reader.load_monthly_performance_guarantee_inputs
load_performance_tests = data_reader.load_performance_tests
write_bess_monthly_results = data_writer.write_bess_monthly_results
input_validation = error_checks.input_validation
generate_bess_invoice_support_report = report.generate_bess_invoice_support_report

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

C:\Users\SamBuck\AppData\Local\Python\pythoncore-3.14-64\python.exe


### Data Inputs and Preprocessing

In [3]:
# Project selector

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

PROJECT_OPTIONS = [("Salinas", "salinas"), ("Jobos", "jobos")]
DEFAULT_PROJECT_ID = "salinas"

if widgets is not None:
    if "project_selector" not in globals():
        project_selector = widgets.ToggleButtons(
            options=PROJECT_OPTIONS,
            value=DEFAULT_PROJECT_ID,
            description="Project:",
            style={"description_width": "initial"},
        )
    display(project_selector)
else:
    project_selector = None

print("Select Salinas or Jobos, then run the next cell.")

ToggleButtons(description='Project:', options=(('Salinas', 'salinas'), ('Jobos', 'jobos')), style=ToggleButton…

Select Salinas or Jobos, then run the next cell.


In [4]:
# Project and filepaths

PROJECT_ID = project_selector.value if project_selector is not None else DEFAULT_PROJECT_ID

PROJECT_DATA_DIR = DATA_DIR / PROJECT_ID
OUTPUT_DIR = PROJECT_DIR / "output" / PROJECT_ID

contract_values_csv = PROJECT_DATA_DIR / "bess_contract_values_template.csv"
yearly_inputs_csv = PROJECT_DATA_DIR / "bess_yearly_inputs_template.csv"
monthly_inputs_csv = PROJECT_DATA_DIR / "bess_monthly_inputs_template.csv"
monthly_performance_guarantee_csv = PROJECT_DATA_DIR / "Monthly_Performance_Guarantee.csv"
performance_tests_csv = PROJECT_DATA_DIR / "Performance_Tests.csv"

monthly_results_csv = OUTPUT_DIR / "bess_monthly_results.csv"
report_txt = OUTPUT_DIR / "report.txt"

print(f"Selected project: {PROJECT_ID}")

Selected project: salinas


In [23]:
# Input file validation

input_validation(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
    monthly_performance_guarantee_csv,
    performance_tests_csv,
)

print("BESS input files passed validation.")

BESS input files passed validation.


In [24]:
# Load contract, yearly, monthly, performance guarantee, and performance test inputs

contract_values, yearly_inputs, monthly_inputs = load_bess_inputs(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
)
monthly_performance_guarantee_inputs = load_monthly_performance_guarantee_inputs(
    monthly_performance_guarantee_csv
)
performance_tests = load_performance_tests(performance_tests_csv)

print(f"Loaded {len(contract_values)} contract value row(s).")
print(f"Loaded {len(yearly_inputs)} yearly input row(s).")
print(f"Loaded {len(monthly_inputs)} monthly input row(s).")
print(f"Loaded {len(monthly_performance_guarantee_inputs)} monthly performance guarantee row(s).")
print(f"Loaded {len(performance_tests)} performance test row(s).")

Loaded 25 contract value row(s).
Loaded 2 yearly input row(s).
Loaded 24 monthly input row(s).
Loaded 2 monthly performance guarantee row(s).
Loaded 1 performance test row(s).


In [16]:
# Input data dashboard

from IPython.display import display

INPUT_CSVS = {
    "Contract values": contract_values_csv,
    "Yearly inputs": yearly_inputs_csv,
    "Monthly inputs": monthly_inputs_csv,
    "Monthly performance guarantee": monthly_performance_guarantee_csv,
    "Performance tests": performance_tests_csv,
}

input_tables = {name: pd.read_csv(path) for name, path in INPUT_CSVS.items()}

contract_values_df = input_tables["Contract values"]
yearly_inputs_df = input_tables["Yearly inputs"]
monthly_inputs_df = input_tables["Monthly inputs"]
monthly_performance_guarantee_df = input_tables["Monthly performance guarantee"]
performance_tests_df = input_tables["Performance tests"]

input_file_summary_df = pd.DataFrame(
    [
        {
            "input": name,
            "rows": len(table),
            "columns": len(table.columns),
            "file": str(path),
        }
        for name, path in INPUT_CSVS.items()
        for table in [input_tables[name]]
    ]
)

display(input_file_summary_df)

if widgets is not None:
    tabs = widgets.Tab()
    children = []
    for name, table in input_tables.items():
        output = widgets.Output()
        with output:
            display(table)
        children.append(output)
    tabs.children = children
    for index, name in enumerate(input_tables):
        tabs.set_title(index, name)
    display(tabs)
else:
    for name, table in input_tables.items():
        print(f"\n{name}")
        display(table)


,input,rows,columns,file
0,Contract values,25,14,C:\Code\salinas-jobos-bess-invoice-calculator\...
1,Yearly inputs,2,4,C:\Code\salinas-jobos-bess-invoice-calculator\...
2,Monthly inputs,24,10,C:\Code\salinas-jobos-bess-invoice-calculator\...
3,Monthly performance guarantee,2,7,C:\Code\salinas-jobos-bess-invoice-calculator\...
4,Performance tests,1,18,C:\Code\salinas-jobos-bess-invoice-calculator\...


### Calculations

In [17]:
# Calculate monthly compensation results

monthly_results = calculate_monthly_results(
    contract_values,
    yearly_inputs,
    monthly_inputs,
    performance_tests,
    monthly_performance_guarantee_inputs,
)

print(f"Calculated {len(monthly_results)} monthly result row(s).")

Calculated 24 monthly result row(s).


In [18]:
# Convert results to a review table

monthly_results_df = pd.DataFrame(
    [
        {
            "timestamp_month": result.timestamp_month,
            "agreement_year": result.agreement_year,
            "CPP": result.cpp,
            "MCC": result.mcc,
            "FA": result.fa,
            "FAA": result.faa,
            "PRA": result.pra,
            "MFP": result.mfp,
            "Other_ADJ": result.other_adj,
            "ALD": result.ald,
            "Actual_Efficiency": result.actual_efficiency,
            "ELD": result.eld,
            "ADJ_Total": result.adj_total,
            "MP": result.mp,
        }
        for result in monthly_results
    ]
)

monthly_results_df

,timestamp_month,agreement_year,CPP,MCC,FA,FAA,PRA,MFP,Other_ADJ,ALD,Actual_Efficiency,ELD,ADJ_Total,MP
0,2026-01,1,24917.00,100.0,0.995924,1.000000,0.995968,2.481653e+06,0.0,0.0,0.970000,0.0,0.0,2.481653e+06
1,2026-02,1,24917.00,100.0,0.994048,1.000000,0.997024,2.484284e+06,250.0,0.0,0.967969,0.0,250.0,2.484034e+06
2,2026-03,1,24917.00,100.0,0.995879,1.000000,0.991935,2.471606e+06,0.0,0.0,NaN,0.0,0.0,2.471606e+06
3,2026-04,1,24917.00,100.0,0.995763,1.000000,0.980556,2.443250e+06,0.0,0.0,NaN,0.0,0.0,2.443250e+06
4,2026-05,1,24917.00,100.0,0.991935,1.000000,0.983871,2.451511e+06,125.0,0.0,NaN,0.0,125.0,2.451386e+06
5,2026-06,1,24917.00,100.0,0.994253,1.000000,0.973611,2.425947e+06,0.0,0.0,NaN,0.0,0.0,2.425947e+06
6,2026-07,1,24917.00,100.0,0.998619,1.000000,0.947581,2.361087e+06,0.0,0.0,NaN,0.0,0.0,2.361087e+06
7,2026-08,1,24917.00,100.0,0.990463,1.000000,0.970430,2.418021e+06,300.0,0.0,NaN,0.0,300.0,2.417721e+06
8,2026-09,1,24917.00,100.0,0.995640,1.000000,0.944444,2.353272e+06,0.0,0.0,NaN,0.0,0.0,2.353272e+06
9,2026-10,1,24917.00,100.0,0.995968,1.000000,0.986559,2.458209e+06,0.0,0.0,NaN,0.0,0.0,2.458209e+06


### Annual Allowance Checks

In [19]:
# Review annual caps and waiting periods used by FA and PRA

allowance_summary_df = (
    monthly_inputs_df
    .groupby("agreement_year", as_index=False)
    .agg(
        POHRS=("POHRS", "sum"),
        GSE=("GSE", "sum"),
        PFM=("PFM", "sum"),
    )
)

allowance_summary_df["POHRS_allowance"] = SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
allowance_summary_df["GSE_waiting_period"] = GRID_SYSTEM_WAITING_PERIOD_HOURS
allowance_summary_df["PFM_waiting_period"] = FORCE_MAJEURE_WAITING_PERIOD_HOURS
allowance_summary_df["POHRS_over_allowance"] = (
    allowance_summary_df["POHRS"] - allowance_summary_df["POHRS_allowance"]
).clip(lower=0)
allowance_summary_df["GSE_over_waiting_period"] = (
    allowance_summary_df["GSE"] - allowance_summary_df["GSE_waiting_period"]
).clip(lower=0)
allowance_summary_df["PFM_over_waiting_period"] = (
    allowance_summary_df["PFM"] - allowance_summary_df["PFM_waiting_period"]
).clip(lower=0)

allowance_summary_df

,agreement_year,POHRS,GSE,PFM,POHRS_allowance,GSE_waiting_period,PFM_waiting_period,POHRS_over_allowance,GSE_over_waiting_period,PFM_over_waiting_period
0,1,168,87,94,160,80,720,8,7,0
1,2,176,72,766,160,80,720,16,0,46


### Write Output Files

In [20]:
# Write monthly result CSV

output_path = write_bess_monthly_results(monthly_results, output_dir=OUTPUT_DIR)
print(f"Wrote monthly results to {output_path}.")

Wrote monthly results to C:\Code\salinas-jobos-bess-invoice-calculator\output\jobos\bess_monthly_results.csv.


### Output Review

In [21]:
# Confirm the generated output file can be read back

written_results_df = pd.read_csv(monthly_results_csv)
written_results_df.head()

,timestamp_month,agreement_year,CPP,MCC,FA,FAA,PRA,MFP,Other_ADJ,ALD,CLD,Actual_Efficiency,ELD,ADJ_Total,MP
0,2026-01,1,24917.0,100.0,0.995924,1.0,0.995968,2.481653e+06,0.0,0.0,0.0,0.970000,0.0,0.0,2.481653e+06
1,2026-02,1,24917.0,100.0,0.994048,1.0,0.997024,2.484284e+06,250.0,0.0,0.0,0.967969,0.0,250.0,2.484034e+06
2,2026-03,1,24917.0,100.0,0.995879,1.0,0.991935,2.471606e+06,0.0,0.0,0.0,NaN,0.0,0.0,2.471606e+06
3,2026-04,1,24917.0,100.0,0.995763,1.0,0.980556,2.443250e+06,0.0,0.0,0.0,NaN,0.0,0.0,2.443250e+06
4,2026-05,1,24917.0,100.0,0.991935,1.0,0.983871,2.451511e+06,125.0,0.0,0.0,NaN,0.0,125.0,2.451386e+06


### Generate Invoice Support Report

This final cell calls `generate_bess_invoice_support_report()` from `report.py`.

In [12]:
# Generate a Section 10.1-style invoice support report using report.py

report_text = generate_bess_invoice_support_report(written_results_df, report_txt)
print(f"Wrote invoice support report to {report_txt}.")
print("\n".join(report_text.splitlines()[:45]))

Wrote invoice support report to C:\Code\salinas-jobos-bess-invoice-calculator\output\salinas\report.txt.
BESS MONTHLY INVOICE SUPPORT REPORT
Source: Section 10.1 Payment & Billings invoice-content requirements
Generated: 2026-06-17 15:03:07

----------------------------------------------------------------------------------------
Billing Period: 2026-01    Agreement Year: 1
----------------------------------------------------------------------------------------

Monthly Fixed Payment
  CPP:  $24,896.00
  MCC:  100.000000 MW
  FA:   99.59%
  FAA:  100.00%
  PRA:  99.60%
  MFP:  $2,479,561.29

Other Payment Adjustments / Credits Owing to PREPA
  Other_ADJ: $0.00
  ALD:       $0.00
  CLD:       $0.00
  Actual Efficiency: 97.00%
  ELD:       $0.00
  ADJ_Total: $0.00

Monthly Payment
  MP = MFP - ADJ_Total
  MP: $2,479,561.29

Not Yet Populated / Itemized Separately
  Discharge Energy and Charge Energy detail: see monthly performance input file
  Ancillary Services: not provided in current i